# 01 · Exploratory Data Analysis
## Bitcoin Transaction Network — Illicit Activity Detection

> **Dataset:** Elliptic Bitcoin Dataset (Elliptic, 2019)  
> **Nodes:** 203,769 transactions | **Edges:** 234,355 directed payment flows  
> **Labels:** 2% illicit · 21% licit · 77% unknown

---
### Research Context
The Elliptic dataset is one of the few publicly available real-world transaction graphs with ground-truth fraud labels. Each node is a Bitcoin transaction; each directed edge represents a Bitcoin flow from one transaction to another. Node features encode transaction-level information across 166 dimensions (94 local + 72 aggregated neighbourhood features).

This notebook establishes the empirical baseline: class imbalance, feature distributions, temporal structure, and graph topology — all of which directly motivate the modelling decisions in subsequent notebooks.

In [ ]:
# ── Environment setup (Colab) ─────────────────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Core
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor' : '#0D1117',
    'axes.facecolor'   : '#161B22',
    'axes.edgecolor'   : '#30363D',
    'axes.labelcolor'  : '#C9D1D9',
    'xtick.color'      : '#8B949E',
    'ytick.color'      : '#8B949E',
    'text.color'       : '#C9D1D9',
    'grid.color'       : '#21262D',
    'grid.linestyle'   : '--',
    'figure.dpi'       : 130,
})
PALETTE = {'Illicit':'#FF4444', 'Licit':'#00C9A7', 'Unknown':'#8B949E'}
print('✅ Environment ready')

In [ ]:
# ── Load Elliptic dataset ─────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set
# Place files in ../data/ or adjust paths below

FEATURES_PATH = '../data/elliptic_txs_features.csv'
CLASSES_PATH  = '../data/elliptic_txs_classes.csv'
EDGES_PATH    = '../data/elliptic_txs_edgelist.csv'

# 166 features: col 0 = txId, col 1 = time_step, cols 2-94 = local, 95-166 = aggregated
feat_cols = ['txId', 'time_step'] + [f'f{i}' for i in range(1, 166)]
features  = pd.read_csv(FEATURES_PATH, header=None, names=feat_cols)
classes   = pd.read_csv(CLASSES_PATH)   # txId, class (1=illicit, 2=licit, unknown)
edges     = pd.read_csv(EDGES_PATH)     # txId1, txId2

# Merge
df = features.merge(classes, on='txId', how='left')
df['class'] = df['class'].replace({'unknown': 0, '1': 1, '2': 2}).astype(float)
df['class_label'] = df['class'].map({1.0:'Illicit', 2.0:'Licit', 0.0:'Unknown'})

print(f'Nodes : {len(df):,}')
print(f'Edges : {len(edges):,}')
print(f'\nClass distribution:')
print(df['class_label'].value_counts())

In [ ]:
# ── 1. Class imbalance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Class Distribution in the Elliptic Dataset', fontsize=14, y=1.02)

counts = df['class_label'].value_counts()
colours = [PALETTE[c] for c in counts.index]

# Bar chart
axes[0].bar(counts.index, counts.values, color=colours, edgecolor='none', width=0.5)
for i, (lbl, val) in enumerate(counts.items()):
    axes[0].text(i, val + 1500, f'{val:,}', ha='center', fontsize=10)
axes[0].set_title('Absolute Counts')
axes[0].set_ylabel('Number of Transactions')

# Pie
axes[1].pie(counts.values, labels=counts.index, colors=colours,
            autopct='%1.1f%%', startangle=140,
            textprops={'color':'#C9D1D9'},
            wedgeprops={'edgecolor':'#0D1117', 'linewidth':2})
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig('../data/fig_class_distribution.png', bbox_inches='tight', dpi=130)
plt.show()

labelled = df[df['class'] != 0]
ill_pct  = (labelled['class']==1).mean()*100
print(f'\nAmong labelled nodes: {ill_pct:.1f}% illicit — severe class imbalance.')
print('→ Motivates: stratified sampling, SMOTE, and F1/AUC as primary metrics.')

In [ ]:
# ── 2. Temporal distribution ──────────────────────────────────────────────────
ts = df.groupby(['time_step','class_label']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Transaction Activity Across 49 Time Steps', fontsize=14)

# Stacked area
for col, colour in PALETTE.items():
    if col in ts.columns:
        axes[0].fill_between(ts.index, ts[col], alpha=0.7, color=colour, label=col)
axes[0].set_ylabel('Node Count')
axes[0].legend(loc='upper right')
axes[0].set_title('All Classes')

# Illicit only — highlight burst windows
if 'Illicit' in ts.columns:
    axes[1].bar(ts.index, ts['Illicit'], color='#FF4444', alpha=0.85, width=0.8)
    # Annotate peak
    peak_ts = ts['Illicit'].idxmax()
    peak_v  = ts['Illicit'].max()
    axes[1].annotate(f'Peak: {peak_v} nodes\n(step {peak_ts})',
                     xy=(peak_ts, peak_v), xytext=(peak_ts+2, peak_v*1.05),
                     arrowprops=dict(arrowstyle='->', color='#FF4444'),
                     color='#FF4444', fontsize=9)
axes[1].set_ylabel('Illicit Node Count')
axes[1].set_xlabel('Time Step')
axes[1].set_title('Illicit Nodes Only — Burst Detection')

plt.tight_layout()
plt.savefig('../data/fig_temporal_distribution.png', bbox_inches='tight', dpi=130)
plt.show()
print('Key finding: illicit activity is NOT uniformly distributed — it occurs in bursts.')
print('→ Motivates temporal features and time-aware train/test splitting.')

In [ ]:
# ── 3. Feature analysis ───────────────────────────────────────────────────────
local_feats = [f'f{i}' for i in range(1, 94)]
labelled    = df[df['class'] != 0].copy()
labelled['label'] = labelled['class'].map({1.0:'Illicit', 2.0:'Licit'})

# Compute mean feature value per class — find most discriminative features
means   = labelled.groupby('label')[local_feats].mean()
diff    = (means.loc['Illicit'] - means.loc['Licit']).abs().sort_values(ascending=False)
top_n   = 10
top_f   = diff.head(top_n).index.tolist()

print(f'Top {top_n} most discriminative local features (by mean difference):')
print(diff.head(top_n).round(4).to_string())

# Box plots for top 6
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Feature Distributions: Illicit vs Licit (Top 6 Discriminative Features)', fontsize=13)

for ax, feat in zip(axes.flat, top_f[:6]):
    for lbl, colour in [('Illicit','#FF4444'), ('Licit','#00C9A7')]:
        vals = labelled[labelled['label']==lbl][feat].dropna()
        # clip extreme outliers for visibility
        p1, p99 = vals.quantile(0.01), vals.quantile(0.99)
        vals = vals.clip(p1, p99)
        ax.hist(vals, bins=40, alpha=0.6, color=colour, label=lbl, density=True)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/fig_feature_distributions.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── 4. Graph topology ─────────────────────────────────────────────────────────
print('Building graph (may take ~30s on full dataset)…')
G = nx.from_pandas_edgelist(edges, source='txId1', target='txId2',
                             create_using=nx.DiGraph())

# Attach labels
label_map = df.set_index('txId')['class_label'].to_dict()
nx.set_node_attributes(G, label_map, 'class_label')

# Degree distributions
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())
df_deg  = df[['txId','class_label']].copy()
df_deg['in_degree']  = df_deg['txId'].map(in_deg).fillna(0).astype(int)
df_deg['out_degree'] = df_deg['txId'].map(out_deg).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Degree Distribution by Class', fontsize=13)

for ax, col, title in zip(axes, ['in_degree','out_degree'], ['In-Degree','Out-Degree']):
    for cls, colour in PALETTE.items():
        vals = df_deg[df_deg['class_label']==cls][col]
        vals = vals[vals > 0]
        if len(vals) == 0: continue
        counts_v = vals.value_counts().sort_index()
        ax.loglog(counts_v.index, counts_v.values, '.', color=colour,
                  alpha=0.6, markersize=4, label=cls)
    ax.set_title(title)
    ax.set_xlabel('Degree (log)')
    ax.set_ylabel('Count (log)')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/fig_degree_distribution.png', bbox_inches='tight', dpi=130)
plt.show()

# Summary stats
print(f'\nGraph summary:')
print(f'  Nodes         : {G.number_of_nodes():,}')
print(f'  Edges         : {G.number_of_edges():,}')
print(f'  Avg in-degree : {np.mean(list(in_deg.values())):.2f}')
print(f'  Avg out-degree: {np.mean(list(out_deg.values())):.2f}')
print(f'  Max in-degree : {max(in_deg.values()):,}')
print(f'  Max out-degree: {max(out_deg.values()):,}')

# Weak connected components
wcc = list(nx.weakly_connected_components(G))
print(f'  WCC count     : {len(wcc):,}')
print(f'  Largest WCC   : {max(len(c) for c in wcc):,} nodes')

In [ ]:
# ── 5. Correlation heatmap (top features) ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Correlation: Illicit vs Licit (Top 15 Features)', fontsize=13)

top15 = diff.head(15).index.tolist()

for ax, lbl, colour_map in zip(axes,
                                ['Illicit', 'Licit'],
                                ['Reds', 'Greens']):
    subset = labelled[labelled['label']==lbl][top15]
    corr   = subset.corr()
    sns.heatmap(corr, ax=ax, cmap=colour_map, center=0,
                linewidths=0.4, linecolor='#0D1117',
                xticklabels=True, yticklabels=True,
                annot=False, fmt='.1f',
                cbar_kws={'shrink':0.8})
    ax.set_title(lbl, color=PALETTE[lbl], fontsize=12)
    ax.tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.savefig('../data/fig_correlation_heatmap.png', bbox_inches='tight', dpi=130)
plt.show()
print('\nEDA complete. Key takeaways:')
print('  1. Severe class imbalance (~10:1 licit:illicit among labelled nodes)')
print('  2. Illicit activity clusters temporally — not randomly distributed')
print('  3. Several local features show strong discriminative power')
print('  4. Graph follows power-law degree distribution (scale-free)')
print('  5. 77% of nodes are unlabelled — semi-supervised approaches warranted')